# Workshop 1: Solutions

## Lab 1: Predict house prices (17:20–17:40)

### Getting started

In [ ]:
import matplotlib.pyplot as plt  # Draws graphs.
import numpy as np  # Does maths on many numbers at once.
import pandas as pd  # Stores data in tables.
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

housing = fetch_california_housing(as_frame=True).frame  # Download the data as a table.
housing.head()  # Show the first five rows.

### 1. Look before you fit

In [ ]:
sample = housing.sample(2000, random_state=0)

incomes = sample["MedInc"]
values = sample["MedHouseVal"]

plt.scatter(incomes, values, s=5)
plt.xlabel("Median income ($10,000s)")
plt.ylabel("Median house value ($100,000s)")
plt.show()

Higher incomes go with higher house values, but the points are widely spread.

The flat line at 5 appears because the census recorded every value above $500,000 as $500,000. Real data often has quirks like this.

### 2. Fit a model and predict

In [ ]:
inputs = housing[["MedInc"]]
outputs = housing["MedHouseVal"]

model = LinearRegression()
model.fit(inputs, outputs)

new_neighbourhoods = pd.DataFrame({"MedInc": [3, 8]})
model.predict(new_neighbourhoods)

The slope is about 0.42: each extra $10,000 of income adds about $42,000 to the predicted house value. The intercept, about 0.45, is the prediction for an income of zero.

In [ ]:
line_ends = pd.DataFrame({"MedInc": [0, 15]})  # Predict at both ends of the x-axis.
plt.scatter(incomes, values, s=5)
plt.plot(line_ends["MedInc"], model.predict(line_ends), color="black")  # Join the two predictions.
plt.xlabel("Median income ($10,000s)")
plt.ylabel("Median house value ($100,000s)")
plt.show()
print("slope:", model.coef_[0], "intercept:", model.intercept_)

### 3. Measure the loss

In [ ]:
actual = outputs
predicted = model.predict(inputs)

errors = actual - predicted
mse = np.mean(errors ** 2)
print(mse)
print(mean_squared_error(actual, predicted))

In [ ]:
baseline = np.full(len(actual), actual.mean())  # The average, once per neighbourhood.
mean_squared_error(actual, baseline)

### 4. Give the model more inputs

In [ ]:
all_inputs = housing.drop(columns="MedHouseVal")

big_model = LinearRegression()
big_model.fit(all_inputs, actual)
mean_squared_error(actual, big_model.predict(all_inputs))

Not yet. We measured the loss on the same data the model trained on. A more flexible model can lower this loss by fitting noise, so we need data the model has not seen. The second half covers this.

### Lab 1 check

- A loss function turns a model’s errors into one number: lower means more accurate predictions.
- A baseline shows whether the model learnt anything useful. An MSE of 0.70 only means something next to the baseline’s 1.33.

### Extension A: Errors in dollars

The MAE is about 0.63, so a typical prediction is about $63,000 away from the true value. The model halves the baseline’s MSE, but it still makes large errors.

In [ ]:
mean_absolute_error(actual, predicted)

### Extension B: Train the model yourself

In [ ]:
x = housing["MedInc"].to_numpy()
y = housing["MedHouseVal"].to_numpy()
slope, intercept = 0.0, 0.0
learning_rate = 0.01

for step in range(1000):
    error = slope * x + intercept - y  # Forward pass, then compare with the answers.
    slope_gradient = 2 * np.mean(error * x)
    intercept_gradient = 2 * np.mean(error)
    slope = slope - learning_rate * slope_gradient
    intercept = intercept - learning_rate * intercept_gradient

print(slope, intercept, np.mean((slope * x + intercept - y) ** 2))

With a learning rate of `0.1`, each step overshoots the minimum by more than the last. The parameters and the loss grow until they overflow to `nan`.

## Take-home: Find the sweet spot (about 15 minutes)

You will choose how flexible a model should be, using validation data.

A **decision tree** predicts by asking a series of yes-or-no questions, such as “Is `MedInc` above 5?”. `max_depth` limits how many questions it can ask in a row. A deeper tree is more flexible.

### T1. Split the data three ways

In [ ]:
train, rest = train_test_split(housing, test_size=0.3, random_state=0)
val, test = train_test_split(rest, test_size=0.5, random_state=0)

input_columns = housing.columns.drop("MedHouseVal")  # The names of the 8 input columns.
print(len(train), len(val), len(test))  # len counts the rows.

### T2. Train trees of increasing depth

In [ ]:
depths = range(1, 21)  # 1, 2, ..., 20
train_losses = []
val_losses = []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
    tree.fit(train[input_columns], train["MedHouseVal"])
    train_losses.append(mean_squared_error(train["MedHouseVal"], tree.predict(train[input_columns])))
    val_losses.append(mean_squared_error(val["MedHouseVal"], tree.predict(val[input_columns])))

plt.plot(depths, train_losses, marker="o", label="Training")
plt.plot(depths, val_losses, marker="o", label="Validation")
plt.xlabel("Tree depth")
plt.ylabel("MSE")
plt.legend()
plt.show()

Shallow trees (depth 1 to about 5) underfit: both losses are high. Deep trees (from about 12) overfit: the training loss falls towards 0, but the validation loss rises again. A depth-20 tree almost memorises the training set.

### T3. Choose a depth, then test it once

In [ ]:
best_position = np.argmin(val_losses)
best_depth = depths[best_position]
print("best depth:", best_depth, "validation MSE:", val_losses[best_position])

best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=0)
best_tree.fit(train[input_columns], train["MedHouseVal"])
mean_squared_error(test["MedHouseVal"], best_tree.predict(test[input_columns]))

The test MSE is about 0.43, close to the validation MSE. That suggests our choice holds up on new data.

If we chose the depth using the test set, we would pick the depth that happens to suit those particular examples. The test score would then look better than the model really is.

### Open-ended: Beat 0.40

There are many good answers. Two models from `sklearn.ensemble` combine many trees and beat 0.40 with their default settings. A random forest reaches about 0.28, and gradient boosting reaches about 0.23. A single tree with no depth limit and `min_samples_leaf=10` also reaches about 0.39: each leaf must average at least 10 neighbourhoods, which stops it memorising single examples.

The random forest’s training MSE (0.04) is far below its validation MSE. That gap shows some overfitting, but we still choose by the lowest validation loss: no other model here does better on new data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

for candidate in [RandomForestRegressor(random_state=0, n_jobs=-1), HistGradientBoostingRegressor(random_state=0)]:
    candidate.fit(train[input_columns], train["MedHouseVal"])
    print(type(candidate).__name__)
    print("training MSE:", mean_squared_error(train["MedHouseVal"], candidate.predict(train[input_columns])))
    print("validation MSE:", mean_squared_error(val["MedHouseVal"], candidate.predict(val[input_columns])))